[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
import torch
import torch.nn as nn

In [30]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        N, D = X.shape
        X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)  # (N, D+1)
        theta = torch.linalg.inv(X_aug.T @ X_aug) @ X_aug.T @ y
        return theta[:D],theta[-1]

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""
        N,D = X.shape
        w = torch.zeros(D)
        b = torch.zeros(())
        for _ in range(steps):
            pred = X @ w + b
            err = pred - y
            grad_W = 2/N * X.T @ err
            grad_b = 2/N * err.sum()
            w -= lr*grad_W
            b -= lr*grad_b
        return w,b

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        D = X.shape[1]
        linear = nn.Linear(D,1)
        optimizer = torch.optim.SGD(linear.parameters(),lr=lr)
        loss_fn = nn.MSELoss()
        for _ in range(steps):
            optimizer.zero_grad()
            pred = linear(X)
            loss = loss_fn(pred,y.unsqueeze(-1))
            loss.backward()
            optimizer.step()
            
        return linear.weight.data.squeeze(0), linear.bias.data

In [31]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [32]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (6.3ms)
  ✅ [2/6] Closed-form finds correct weights (3.6ms)
  ✅ [3/6] Gradient descent converges (60.7ms)
  ✅ [4/6] nn.Linear approach works (130.2ms)
  ✅ [5/6] All three methods agree (240.6ms)
  ✅ [6/6] Closed-form uses no autograd (0.5ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (441.9ms total)
  Progress saved. Run status() to see your dashboard.

